<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Tiled GEMM

Matrix multiply, `C = A @ B`, is *the* workload GPUs are built for, and the perfect lens for the
single most important GPU performance idea: **reuse data in fast memory instead of re-reading it
from slow memory.**

We write the kernel twice. First a **naive** version — one thread per output element — to pin
down the data flow and check it against PyTorch. It is correct but bandwidth-bound: every thread
reads a whole row of A and column of B straight from global memory. Then we stage tiles in
**shared memory**, so each element is fetched once per block instead of once per output element.

That rewrite is your first use of `cutlass.Array(..., space=smem)` and CTA barriers — the
load -> sync -> compute -> sync loop at the heart of every fast GPU kernel.

**You'll learn:** mapping a 2-D thread grid onto an output matrix; flat row-major indexing into
`cutlass.Array` kernel args; allocating *shaped* shared-memory tiles for 2-D access; the
cooperative-load pattern; and why the load -> sync -> compute -> sync loop needs a CTA barrier
(`cute.arch.barrier()`) on *both* ends.

**Runs on:** any CUDA GPU. **Prereq:** the `01_array_concepts` notebook.

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. The naive version (baseline)

The simplest mapping: **one thread per output element** `C[i, j]`. Each thread reads its `(i, j)`
from the grid, walks the whole K dimension accumulating the dot product, and writes one result.
Get this working first — it pins down the indexing math and gives a correct reference for the
faster kernel.

The matrices arrive as flat `cutlass.Array`s with no layout attached, so — just like in CUDA C —
you turn 2-D coordinates into a flat row-major offset by hand: `A[i,k] -> a[i*K+k]`,
`B[k,j] -> b[k*N+j]`, `C[i,j] -> c[i*N+j]`.

In [ ]:
@cute.kernel
def naive_gemm_kernel(
    a: cutlass.Array,
    b: cutlass.Array,
    c: cutlass.Array,
    M: cutlass.Int32,
    N: cutlass.Int32,
    K: cutlass.Int32,
    TS: cutlass.Constexpr,  # unused here; the shared host entry passes the tile size
):
    tx, ty, _ = cute.arch.thread_idx()
    bx, by, _ = cute.arch.block_idx()
    bdx, bdy, _ = cute.arch.block_dim()

    # Step 1. This thread owns output element C[row, col].
    row = bx * bdx + tx
    col = by * bdy + ty

    if row < M and col < N:
        # Step 2. Dot product over K. The cost: this re-reads a full row of A and a
        # full column of B from global memory for every single output element.
        acc = 0.0
        for k in range(K):
            acc += a[row * K + k] * b[k * N + col]
        # Step 3. Write the one output element this thread owns.
        c[row * N + col] = acc

## 2. The problem, and the fix: tile into shared memory

In the naive kernel, neighbouring threads re-read the *same* rows of A and columns of B from
**global** memory over and over. Global memory is large but slow; the same bytes cross the bus
hundreds of times.

**Shared memory** (SMEM) is a small, fast scratchpad shared by every thread in a block. The fix:
the block cooperatively copies a `TS x TS` tile of A and of B into SMEM *once*, then all threads
compute against that fast copy. We march along K one tile at a time, and for each tile:

| step | what happens |
|---|---|
| **1. cooperative load** | each thread copies ONE element of the A-tile and the B-tile into SMEM |
| **2. barrier** | wait until the whole tile is in SMEM before anyone reads it |
| **3. compute** | every thread multiplies the two SMEM tiles into its accumulator |
| **4. barrier** | wait until everyone is done before the next load overwrites the tile |

Each global element is now read **once per block** instead of once per output element — that is the
whole win. Note the two `cutlass.Array` flavors: the **global** matrices stay flat (indexed 1-D),
while the **SMEM tiles** are allocated *with a shape* `(TS, TS)` and indexed 2-D as `a_smem[ty, tx]`.

In [ ]:
@cute.kernel
def tiled_gemm_kernel(
    a: cutlass.Array,
    b: cutlass.Array,
    c: cutlass.Array,
    M: cutlass.Int32,
    N: cutlass.Int32,
    K: cutlass.Int32,
    TS: cutlass.Constexpr,
):
    tx, ty, _ = cute.arch.thread_idx()
    bx, by, _ = cute.arch.block_idx()

    # Per-block scratchpads. Shaping them (TS, TS) lets us index 2-D below.
    a_smem = cutlass.Array(cutlass.Float32, (TS, TS), space=cutlass.AddressSpace.smem)
    b_smem = cutlass.Array(cutlass.Float32, (TS, TS), space=cutlass.AddressSpace.smem)

    # This thread owns one output element; (row, col) is its position in C.
    row = bx * TS + ty
    col = by * TS + tx

    acc = 0.0
    for k_tile in range(0, K, TS):
        # Step 1. Cooperative load: every thread brings in ONE element of each tile, so
        # together the block stages the full TS x TS tiles of A and B into SMEM.
        a_smem[ty, tx] = a[row * K + (k_tile + tx)]
        b_smem[ty, tx] = b[(k_tile + ty) * N + col]
        # Step 2. Don't read the tiles until every thread has finished writing them.
        cute.arch.barrier()

        # Step 3. Multiply the two SMEM tiles into the accumulator (fast reads, no global traffic).
        for kk in range(TS):
            acc += a_smem[ty, kk] * b_smem[kk, tx]
        # Step 4. Don't overwrite the tiles (next iteration's load) until everyone is done reading.
        cute.arch.barrier()

    # Step 5. Write this thread's accumulated output element.
    c[row * N + col] = acc

## 3. Launch and verify

One block computes one `TS x TS` output tile, so the block shape *is* the tile `(TS, TS, 1)` and
the grid tiles all of C. For simplicity we assume M, N, K are multiples of `TS` (true here: 1024
with `TS=32`) — production kernels mask the ragged edge tiles instead. A small `make_gemm`
**factory** bakes the kernel variant and tile size into a ready-to-call launcher, so each kernel
gets its own `gemm` with `TS` already baked in. `cutlass.Array` parameters accept the PyTorch CUDA
tensors **directly** via `cute.runtime.from_dlpack`.

In [ ]:
def make_gemm(kernel, TS=32):
    """Factory: bake the kernel variant + tile size into a ready-to-call GEMM host.

    `kernel` and `TS` are captured at build time, so the returned host takes only the
    matrices and the runtime `M, N, K` -- one specialized launcher per kernel variant.
    """

    @cute.jit
    def gemm(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array,
             M: cutlass.Int32, N: cutlass.Int32, K: cutlass.Int32):
        # One block per output tile. M, N, K stay runtime; TS is baked in, so the grid
        # and block shapes are compile-time constants.
        grid = (M // TS, N // TS, 1)
        kernel(a, b, c, M, N, K, TS).launch(grid=grid, block=(TS, TS, 1))

    return gemm

In [ ]:
M, N, K = 256, 256, 256
TS = 32  # output-tile size, and also the block dimension (see the tiled kernel)
# The grid and the K-loop both step by TS, so M, N, K must divide evenly by it —
# otherwise the remainder tiles would be silently dropped (no edge masking here).
assert M % TS == 0 and N % TS == 0 and K % TS == 0, "M, N, K must each be a multiple of TS"
a = torch.randn(M, K, dtype=torch.float32, device="cuda")
b = torch.randn(K, N, dtype=torch.float32, device="cuda")
ref = a.cpu() @ b.cpu()

# Build one specialized launcher per kernel variant — TS baked in by the factory, the
# matrices and runtime M, N, K passed at the call.
naive = make_gemm(naive_gemm_kernel, TS)
tiled = make_gemm(tiled_gemm_kernel, TS)

for name, run in (("naive", naive), ("tiled", tiled)):
    c = torch.zeros(M, N, dtype=torch.float32, device="cuda")
    run(cute.runtime.from_dlpack(a), cute.runtime.from_dlpack(b), cute.runtime.from_dlpack(c), M, N, K)
    torch.testing.assert_close(c.cpu(), ref, atol=1e-3, rtol=1e-3)
    print(f"PASS  {name}")

# Expected output:
# PASS  naive
# PASS  tiled

## Try it yourself

1. Set `TS = 16` and rerun — still correct? How do the block and grid shapes change?
2. Delete the **second** `cute.arch.barrier()` and rerun a few times — the result goes wrong
   intermittently. Why? (A fast thread races ahead and overwrites `a_smem` while a slower thread is
   still reading the previous tile.)
3. Count the global loads per output element for the naive vs. the tiled kernel. Where did the
   factor of `TS` reduction come from?